In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVR # Keep SVR for regression, remove SVC import
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score, f1_score, roc_auc_score
import xgboost as xgb
import os
import pickle

# --- Configuration ---
train_file = 'train_processed_final.csv'  # Path to preprocessed training data
test_file = 'test_processed_final.csv'    # Path to preprocessed test data
sample_submission_file = 'sample_submission.csv' # To get the correct ID format/output structure
submission_dir = 'submissions'            # Directory to save predictions
tuning_results_dir = 'tuning_results'     # Directory where tuned models are saved
# target_column = 'target'  # Removed fixed target column name
N_SPLITS = 5 # Number of folds for cross-validation

# --- Ensure submission directory exists ---
os.makedirs(submission_dir, exist_ok=True)

# --- Load Tuned Hyperparameters (if available) ---
tuned_params = {}
tuning_results_summary_path = os.path.join(tuning_results_dir, 'tuning_results_summary.txt')
if os.path.exists(tuning_results_summary_path):
    print(f"Loading tuned hyperparameters from {tuning_results_summary_path}")
    try:
        # --- NEW APPROACH: Load Best Model Objects ---
        best_model_files = [f for f in os.listdir(tuning_results_dir) if f.startswith('best_model_') and f.endswith('.pkl')]
        for model_file in best_model_files:
            model_name = model_file.replace('best_model_', '').replace('.pkl', '').capitalize()
            # Exclude SVC if it was saved
            if model_name == 'SVC':
                 print(f"Skipping loaded model for {model_name} (removed from pipeline).")
                 continue
            model_path = os.path.join(tuning_results_dir, model_file)
            try:
                with open(model_path, 'rb') as f_model:
                     loaded_model = pickle.load(f_model)
                     tuned_params[model_name] = loaded_model # Store the full fitted model object
                     print(f"Loaded best model object for {model_name} from {model_file}")
            except Exception as e:
                 print(f"Could not load best model from {model_file}: {e}")

    except Exception as e:
        print(f"Could not read tuning results summary {tuning_results_summary_path}: {e}")
        print("Will use default model parameters.")
else:
    print(f"Tuning results file {tuning_results_summary_path} not found. Will use default model parameters.")


# --- Load Data ---
print("Loading data...")
try:
    if os.path.exists(train_file):
        df_train = pd.read_csv(train_file)
        print(f"Loaded training  {df_train.shape}")
    else:
        print(f"Error: {train_file} not found. Please ensure preprocessed data is available.")
        exit()

    if os.path.exists(test_file):
        df_test = pd.read_csv(test_file)
        print(f"Loaded test  {df_test.shape}")
    else:
        print(f"Error: {test_file} not found. Please ensure preprocessed test data is available.")
        exit()

    if os.path.exists(sample_submission_file):
        sample_submission = pd.read_csv(sample_submission_file)
        print(f"Loaded sample submission: {sample_submission.shape}")
    else:
        print(f"Warning: {sample_submission_file} not found. Using test set index for submission.")
        sample_submission = None

except FileNotFoundError as e:
    print(f"File not found: {e}")
    exit()

# --- Prepare Features and Target ---
# Identify the target column as the LAST column in the training set
target_column = df_train.columns[-1]  # Get the name of the last column
print(f"Identified target column: '{target_column}'")

if target_column not in df_train.columns:
    print(f"Error: Target column '{target_column}' (assumed to be the last column) not found in training data.")
    print(f"Available columns: {list(df_train.columns)}")
    exit()

X = df_train.drop(columns=[target_column])
y = df_train[target_column]

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")

# --- Determine Feature Names from Training Set ---
feature_names = X.columns.tolist()
print(f"Feature names for training/prediction: {feature_names}")

# --- Identify the ID column for submission ---
# Use the ID from sample submission if available, otherwise use test set index
if sample_submission is not None and len(sample_submission.columns) > 0:
    id_column = sample_submission.columns[0] # Assuming the first column is the ID
    # Load the ID values from the sample submission file
    test_ids = sample_submission[id_column]
    print(f"Using ID column '{id_column}' from sample submission.")
else:
    # Use the test set's index as the ID if no sample submission is provided
    # Check if 'Id' exists in test set, otherwise use index
    if 'Id' in df_test.columns:
        test_ids = df_test['Id']
        print(f"Using 'Id' column from test set.")
    else:
        test_ids = df_test.index
        id_column = 'index' # Placeholder name for index if needed later
        print(f"Using test set index as ID column.")

# --- Prepare Test Set Features ---
# Remove the target column (if it exists) and any non-feature columns (like Id) from the test set
# Only keep the columns that match the training features
# Check if the expected features exist in the test set
missing_features = [col for col in feature_names if col not in df_test.columns]
if missing_features:
    print(f"Error: The following features from the training set are missing in the test set: {missing_features}")
    exit()

# Select only the relevant features for prediction
X_test = df_test[feature_names] # This ensures only training features are used

print(f"Test features (X_test) shape: {X_test.shape}")

# --- Determine Problem Type (Classification vs Regression) ---
# Check the data type of the target variable
if y.dtype == 'object' or isinstance(y.dtype, pd.CategoricalDtype) or len(y.unique()) <= 20: # Heuristic for classification, updated deprecation warning
    problem_type = 'classification'
    print(f"Detected target type as categorical/discrete. Assuming {problem_type}.")

    # Define default models (excluding SVC)
    default_models = {
        'NaiveBayes': GaussianNB(),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
        'XGB': xgb.XGBClassifier(random_state=42),
        'KNN': KNeighborsClassifier(n_neighbors=5)
        # Removed SVC
    }

    # Use tuned models if available, otherwise use defaults
    model_definitions = {}
    for name, default_model in default_models.items():
        if name in tuned_params:
            # Use the loaded best model object directly
            print(f"  Using tuned model object for {name}, refitting on full training data.")
            model_definitions[name] = tuned_params[name] # The loaded best model object
            # Refit the loaded model on the full training set
            model_definitions[name].fit(X, y) # Refit on full X, y
            print(f"  Refitted tuned model for {name} on full training data.")
        else:
            print(f"  Using default model for {name}.")
            model_definitions[name] = default_model

    scoring = ['accuracy', 'f1_macro', 'roc_auc_ovr'] # Scoring metrics for classification
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # Use Stratified KFold for classification
else:
    problem_type = 'regression'
    print(f"Detected target type as numeric/continuous. Assuming {problem_type}.")
    # Define default regression models
    model_definitions = {
        'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
        'SVR': SVR(), # Keep SVR for regression
        'XGB': xgb.XGBRegressor(random_state=42),
        'KNN': KNeighborsRegressor(n_neighbors=5)
        # Removed Naive Bayes for regression (less common, potentially inappropriate)
    }
    scoring = ['neg_mean_squared_error', 'r2'] # Scoring metrics for regression
    cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42) # Use regular KFold for regression


# --- Train Models (or use pre-tuned and refitted ones), Perform Cross-Validation, and Generate Predictions ---
for name, model in model_definitions.items():
    print(f"--- Training/Using {name} (Cross-Validating if default) ---")
    try:
        # Check if this model was loaded from tuning (and thus already fitted on a part of the data)
        # If it was loaded, we already refitted it on the full training data above.
        # If it's a default model, we need to fit it now.
        if name not in tuned_params:
             # It's a default model, fit it now
             print(f"  Fitting default {name} on full training set...")
             model.fit(X, y)
        # If it was loaded, it's already refitted on the full set as per the logic above.

        # Perform cross-validation *only* for default models or if desired for the refitted tuned model
        # Cross-validation on the *final* refitted model using the full training data X, y
        # is less standard, as the hyperparameters were already chosen via CV on a subset.
        # Usually, CV is done during hyperparameter tuning.
        # Let's skip CV here if the model was loaded from tuning, or perform it for consistency.
        # Performing CV on the final model fitted on full data gives an estimate of performance
        # on unseen data, assuming the model is stable.
        # Let's perform CV for all models for consistency in reporting.
        print(f"  Performing {N_SPLITS}-Fold Cross-Validation on the final fitted model...")
        cv_results = {}
        for metric in scoring:
            scores = cross_val_score(model, X, y, cv=cv, scoring=metric)
            cv_results[metric] = {
                'mean': scores.mean(),
                'std': scores.std()
            }
            print(f"    {metric}: Mean={cv_results[metric]['mean']:.4f}, Std={cv_results[metric]['std']:.4f}")


        print(f"  Predicting on test set with {name}...")
        # Use the correctly formatted test features (X_test)
        predictions = model.predict(X_test)

        # Create submission dataframe
        submission_df = pd.DataFrame({
            id_column: test_ids, # Use the ID column determined earlier
            'Class': predictions # Use 'target' as the column name for predictions in submission
        })

        # Save predictions to CSV in the submission directory
        submission_path = os.path.join(submission_dir, f'submission_{name.lower()}.csv')
        submission_df.to_csv(submission_path, index=False)
        print(f"  Predictions for {name} saved to {submission_path}")

    except Exception as e:
        print(f"  Error training/cross-validating/predicting with {name}: {e}")

print("All models processed (trained/tuned), cross-validated, and predictions saved (if successful). Check the 'submissions' folder.")

Loading tuned hyperparameters from tuning_results\tuning_results_summary.txt
Loaded best model object for Knn from best_model_knn.pkl
Loaded best model object for Randomforest from best_model_randomforest.pkl
Loaded best model object for Xgb from best_model_xgb.pkl
Loading data...
Loaded training  (14396, 15)
Loaded test  (3600, 15)
Loaded sample submission: (3600, 2)
Identified target column: 'Class'
Features (X) shape: (14396, 14)
Target (y) shape: (14396,)
Feature names for training/prediction: ['Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_in min/ms', 'time_signature']
Using ID column 'Id' from sample submission.
Test features (X_test) shape: (3600, 14)
Detected target type as categorical/discrete. Assuming classification.
  Using default model for NaiveBayes.
  Using default model for RandomForest.
  Using default model for XGB.
  Using default model for KNN.
--- Train